In [ ]:
!git clone https://github.com/binance/binance-public-data.git

%cd binance-public-data/python

!mkdir -p /content/binance_data

!python download-kline.py -t spot -s BTCUSDT -y 2023 -folder /content/binance_data -skip-daily 1

%cd /content

y
Cloning into 'binance-public-data'...



remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 282 (delta 100), reused 49 (delta 49), pack-reused 153 (from 2)
Receiving objects: 100% (282/282), 6.51 MiB | 13.25 MiB/s, done.
Resolving deltas: 100% (149/149), done.
/content/binance-public-data/python
Folder already exists! Do you want to overwrite it? y/n  y
Found 1 symbols
[1/1] - start download monthly BTCUSDT klines 

File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1s/BTCUSDT-1s-2023-01.zip
[##################################################]
File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1s/BTCUSDT-1s-2023-02.zip
[##################################################]
File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1s/BTCUSDT-1s-2023-03.zip
[##################################################]
File Download: /content/binance

In [ ]:
# 1. Move back into the downloader folder
%cd /content/binance-public-data/python

# 2. Download the MINUTE data for 2023
!python download-kline.py -t spot -s BTCUSDT -y 2023 -i 1m -folder /content/binance_data -skip-daily 1

# 3. Return to root
%cd /content

/content/binance-public-data/python
Folder already exists! Do you want to overwrite it? y/n  y
Found 1 symbols
[1/1] - start download monthly BTCUSDT klines 

File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-2023-01.zip
[##################################################]
File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-2023-02.zip
[##################################################]
File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-2023-03.zip
[##################################################]
File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-2023-04.zip
[##################################################]
File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-2023-05.zip
[##################################################]
File Download: /content/binance_data/data/spot/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-2023-06.zip


In [ ]:
!pip install hmmlearn -q

import pandas as pd
import numpy as np
import glob
import time
import warnings
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")

file_path = '/content/binance_data/data/spot/monthly/klines/BTCUSDT/1m/*.zip'
all_files = glob.glob(file_path)
all_files.sort()

columns = [
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_asset_volume', 'number_of_trades',
    'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'
]

print("Loading massive 1-minute zip files...")
df = pd.concat((pd.read_csv(f, names=columns) for f in all_files), ignore_index=True)
df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
df.set_index('open_time', inplace=True)
df['close'] = df['close'].astype(float)

# Calculate and scale returns
df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
df = df.dropna().copy()
scaled_returns = df['log_returns'] * 100

print(f"Final Dataset Size: {len(df):,} rows.")
print("🚨 Fitting Production HMM via hmmlearn (This will be much faster)...")

start_time = time.time()

# 1. Reshape data for hmmlearn (requires 2D array)
X = scaled_returns.values.reshape(-1, 1)

# 2. Initialize the Gaussian Hidden Markov Model
# covariance_type="diag" prevents the matrix inversion crashes entirely
model = GaussianHMM(n_components=2, covariance_type="diag", n_iter=50, random_state=42)



from scipy.special import logsumexp

# ... (Previous code: Loading data, scaling returns) ...

# 3. Fit the model to the half-million rows
model.fit(X)

print("🚨 Executing Custom Causal Forward-Pass (Preventing Data Leakage)...")

# Extract the learned transition and starting probabilities
log_transmat = np.log(np.maximum(model.transmat_, 1e-20))
log_startprob = np.log(np.maximum(model.startprob_, 1e-20))

# 4. Extract the raw emission probabilities (log-likelihoods) for each minute
framelogprob = model._compute_log_likelihood(X)
n_samples, n_components = framelogprob.shape

# Initialize the Forward Lattice (alpha) matrix
fwdlattice = np.zeros((n_samples, n_components))
fwdlattice[0] = log_startprob + framelogprob[0]

# 5. The Recursive Forward Algorithm (Strictly causal, past-data only)
# alpha_t(j) = logsumexp(alpha_{t-1}(i) + Transition(i -> j)) + Emission_t(j)
for t in range(1, n_samples):
    fwdlattice[t] = logsumexp(fwdlattice[t-1][:, np.newaxis] + log_transmat, axis=0) + framelogprob[t]

# 6. Convert the log-space lattice into normalized probabilities (0.0 to 1.0)
filtered_probs = np.exp(fwdlattice - logsumexp(fwdlattice, axis=1, keepdims=True))

# 7. Lock in the causal regime state
causal_hidden_states = np.argmax(filtered_probs, axis=1)

elapsed = (time.time() - start_time) / 60
print(f"✅ HMM Converged and Filtered in {elapsed:.2f} minutes!")

# Extract the regimes and save the CSV
df['regime_state'] = causal_hidden_states
df.to_csv('/content/binance_btc_1m_with_regimes.csv')
print("✅ Saved Causal 'binance_btc_1m_with_regimes.csv' for PyTorch!")

Loading massive 1-minute zip files...
Final Dataset Size: 525,519 rows.
🚨 Fitting Production HMM via hmmlearn (This will be much faster)...
🚨 Executing Custom Causal Forward-Pass (Preventing Data Leakage)...
✅ HMM Converged and Filtered in 1.63 minutes!
✅ Saved Causal 'binance_btc_1m_with_regimes.csv' for PyTorch!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np
import logging
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class TimeSeriesReturnDataset(Dataset):
    def __init__(self, data_path: str, seq_length: int = 60):
        self.seq_length = seq_length
        self.features = ['open', 'high', 'low', 'close', 'volume', 'log_returns', 'regime_state']

        logger.info(f"Loading High-Frequency Dataset from {data_path}...")
        df = pd.read_csv(data_path).dropna(subset=self.features).reset_index(drop=True)

        self.raw_close = df['close'].values
        self.scaler = StandardScaler()
        self.scaled_data = self.scaler.fit_transform(df[self.features].values)
        logger.info(f"Dataset successfully loaded and scaled: {len(self.scaled_data):,} rows.")

    def __len__(self) -> int:
        return len(self.scaled_data) - self.seq_length

    def __getitem__(self, idx: int):
        x_sequence = self.scaled_data[idx : idx + self.seq_length]
        y_target = self.scaled_data[idx + self.seq_length, 5] # Predict log_returns
        return torch.tensor(x_sequence, dtype=torch.float32), torch.tensor(y_target, dtype=torch.float32)

class SelectiveStateSpaceBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = self.d_model * expand

        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(in_channels=self.d_inner, out_channels=self.d_inner, kernel_size=d_conv, groups=self.d_inner, padding=d_conv - 1)
        self.act = nn.SiLU()
        self.dt_act = nn.Softplus()

        self.x_proj = nn.Linear(self.d_inner, self.d_state * 2 + self.d_inner, bias=False)
        self.dt_proj = nn.Linear(self.d_inner, self.d_inner, bias=True)

        self.A_log = nn.Parameter(torch.log(torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape
        projected = self.in_proj(x)
        x_branch, res_branch = projected.split(self.d_inner, dim=-1)

        x_conv = x_branch.transpose(1, 2)
        x_conv = self.conv1d(x_conv)[:, :, :seq_len]
        x_conv = self.act(x_conv.transpose(1, 2))

        x_proj_out = self.x_proj(x_conv)
        dt, B, C = torch.split(x_proj_out, [self.d_inner, self.d_state, self.d_state], dim=-1)
        dt = self.dt_act(self.dt_proj(dt))

        A = -torch.exp(self.A_log)
        y_seq = []
        h = torch.zeros(batch, self.d_inner, self.d_state, device=x.device)

        for t in range(seq_len):
            dt_t, B_t, C_t, x_t = dt[:, t, :].unsqueeze(-1), B[:, t, :].unsqueeze(1), C[:, t, :].unsqueeze(-1), x_conv[:, t, :].unsqueeze(-1)
            h = torch.exp(dt_t * A.unsqueeze(0)) * h + (dt_t * B_t) * x_t.expand(-1, -1, self.d_state)
            y_seq.append(torch.bmm(h, C_t).squeeze(-1))

        y = torch.stack(y_seq, dim=1) * self.act(res_branch)
        return self.out_proj(y)

class RegimeAwareMambaForecaster(nn.Module):
    def __init__(self, input_dim: int = 7, d_model: int = 64, d_state: int = 16):
        super().__init__()
        self.feature_extractor = nn.Linear(input_dim, d_model)
        self.state_space_block = SelectiveStateSpaceBlock(d_model=d_model, d_state=d_state)
        self.regression_head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.feature_extractor(x)
        return self.regression_head(self.state_space_block(x)[:, -1, :]).squeeze(-1)

if __name__ == "__main__":
    # Pointing to the new HFT dataset
    DATA_PATH = '/content/binance_btc_1m_with_regimes.csv'
    dataset = TimeSeriesReturnDataset(DATA_PATH, seq_length=60)

    train_size = int(len(dataset) * 0.8)
    train_dataset = Subset(dataset, range(0, train_size))
    test_dataset = Subset(dataset, range(train_size, len(dataset)))

    # Massive batch size for HFT data speed
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = RegimeAwareMambaForecaster(input_dim=7).to(device)

    class RealTimeHFTLoss(nn.Module):
        def __init__(self, direction_penalty=5.0):
            super().__init__()
            self.mse = nn.MSELoss()
            self.direction_penalty = direction_penalty

        def forward(self, predictions, targets):
            base_error = self.mse(predictions, targets)

            # Extract the mathematical signs (+ for UP, - for DOWN)
            pred_signs = torch.sign(predictions)
            target_signs = torch.sign(targets)

            # Massive penalty if the model guesses the wrong direction or tries to flatline
            sign_mismatch = (pred_signs != target_signs).float()
            directional_loss = torch.mean(sign_mismatch) * self.direction_penalty

            return base_error + directional_loss

    # Initialize the new strict, real-time loss function
    criterion = RealTimeHFTLoss(direction_penalty=5.0)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 5
    logger.info(f"Training on {device.type.upper()} predicting HFT Log Returns...")

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        start_time = time.time()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        elapsed = time.time() - start_time
        logger.info(f"Epoch {epoch+1:02d}/{epochs} | Loss (MSE): {total_loss/len(train_loader):.4f} | Time: {elapsed:.1f}s")

    logger.info("Executing evaluation and Absolute Price Reconstruction...")
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            all_preds.extend(model(X_batch.to(device)).cpu().numpy())
            all_targets.extend(y_batch.numpy())

    dummy_array_preds = np.zeros((len(all_preds), 7))
    dummy_array_preds[:, 5] = all_preds
    predicted_log_returns = dataset.scaler.inverse_transform(dummy_array_preds)[:, 5]

    start_idx = train_size + dataset.seq_length
    t_minus_1_prices = dataset.raw_close[start_idx - 1 : start_idx - 1 + len(predicted_log_returns)]
    actual_prices = dataset.raw_close[start_idx : start_idx + len(predicted_log_returns)]

    predicted_prices = t_minus_1_prices * np.exp(predicted_log_returns)

    mae = mean_absolute_error(actual_prices, predicted_prices)
    rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))

    print("\n" + "="*50)
    print(" HIGH-FREQUENCY HFT METRICS      ")
    print("="*50)
    print(f"Mean Absolute Error (MAE):      ${mae:.4f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:.4f}")
    print("="*50 + "\n")


 HIGH-FREQUENCY HFT METRICS      
Mean Absolute Error (MAE):      $14.8233
Root Mean Squared Error (RMSE): $23.4877



In [ ]:
import torch
import joblib

# 1. Save the HFT PyTorch Model Weights
MODEL_PATH = '/content/hft_mamba_weights.pth'
torch.save(model.state_dict(), MODEL_PATH)

# 2. Save the 7-Feature Scaler
SCALER_PATH = '/content/hft_scaler.pkl'
joblib.dump(dataset.scaler, SCALER_PATH)

print(f"✅ HFT Model weights saved to: {MODEL_PATH}")
print(f"✅ HFT Scaler object saved to: {SCALER_PATH}")

✅ HFT Model weights saved to: /content/hft_mamba_weights.pth
✅ HFT Scaler object saved to: /content/hft_scaler.pkl


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error, r2_score

# Ensure we have the MAE from the previous run to compare
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(actual_prices, predicted_prices)

# 1. Mean Absolute Percentage Error (MAPE)
mape = mean_absolute_percentage_error(actual_prices, predicted_prices) * 100

# 2. R-squared (R2) Score
r2 = r2_score(actual_prices, predicted_prices)

# 3. Directional Accuracy (Hit Rate) - The Ultimate Test
# Did the model correctly predict if the next minute was going UP or DOWN?
actual_deltas = np.diff(actual_prices)
predicted_deltas = np.diff(predicted_prices)

# Compare signs (positive for up, negative for down)
# We ignore 0s (no price movement) to be strictly rigorous
valid_indices = (actual_deltas != 0)
correct_directions = np.sum(np.sign(actual_deltas[valid_indices]) == np.sign(predicted_deltas[valid_indices]))
directional_accuracy = (correct_directions / len(actual_deltas[valid_indices])) * 100

# 4. The Naive Baseline Check (The "Shift" Test)
# What if a dumb model just copies the T-1 price?
naive_predictions = actual_prices[:-1]
naive_actuals = actual_prices[1:]
naive_mae = mean_absolute_error(naive_actuals, naive_predictions)

print("\n" + "="*50)
print("      PHASE 2.5: ADVANCED OVERFIT DIAGNOSTICS      ")
print("="*50)
print(f"1. MAPE (Percentage Error):      {mape:.4f}%")
print(f"2. R-Squared Score (Fit):        {r2:.4f}")
print(f"3. Directional Accuracy:         {directional_accuracy:.2f}%")
print("-" * 50)
print("      RANDOM WALK vs. MAMBA BATTLE                 ")
print("-" * 50)
print(f"Naive 'Copycat' MAE:             ${naive_mae:.4f}")
print(f"Mamba Neural Network MAE:        ${mae:.4f}")
print("="*50 + "\n")

if mae < naive_mae:
    print("✅ SUCCESS: Mamba mathematically beat the Random Walk baseline!")
else:
    print("⚠️ WARNING: Mamba is suffering from the 1-Step Lag (Overfitting to T-1).")


      PHASE 2.5: ADVANCED OVERFIT DIAGNOSTICS      
1. MAPE (Percentage Error):      0.0386%
2. R-Squared Score (Fit):        1.0000
3. Directional Accuracy:         52.18%
--------------------------------------------------
      RANDOM WALK vs. MAMBA BATTLE                 
--------------------------------------------------
Naive 'Copycat' MAE:             $14.7554
Mamba Neural Network MAE:        $14.8233

⚠️ WARNING: Mamba is suffering from the 1-Step Lag (Overfitting to T-1).
